# 10 — Does campaign money predict electoral support? (2024)

**Question:** Are candidates who raise or spend more money also mentioned more
often and ranked first more often?

This is the top-line notebook. It deliberately uses only four variables:

- total fundraising;
- total spending;
- ballot mentions;
- first-place votes.

We use correlations to describe the relationship and simple OLS regressions
to compare **raw dollars versus log dollars**.

Interactive Plotly figures let us hover over every point and identify the
candidate.

> Association is not causation.


## 1. Setup

In [37]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 100)

# Find the repository root from either the repo root or notebooks/.
cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


## 2. Build one candidate-level table

Ballot support is the left-hand table. A candidate stays in the analysis even
when a finance record is missing; missing finance remains `NaN`, not zero.


In [38]:
support_path = (
    PROCESSED
    / "ballot_support"
    / str(YEAR)
    / "candidate_ballot_support_2024.csv"
)

fundraising_path = (
    fundraising_processed_dir(YEAR, CONTEST)
    / "openelections_candidate_fundraising_summary.csv"
)

spending_path = (
    spending_processed_dir(YEAR, CONTEST)
    / "orestar_candidate_spending_summary.csv"
)

support = pd.read_csv(support_path)
fundraising = pd.read_csv(fundraising_path)
spending = pd.read_csv(spending_path)

# Keep only spending records linked to an official candidate.
spending = spending[
    spending["candidate_key"].notna()
].copy()

# Add spending across filings if a candidate ever has more than one.
spending = (
    spending
    .groupby("candidate_key", as_index=False)
    .agg(
        total_spending=("total_spending", "sum"),
        expenditure_count=("expenditure_count", "sum"),
    )
)

analysis = support.merge(
    fundraising[
        [
            "candidate_key",
            "total_amount",
            "total_contribution_count",
        ]
    ],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

analysis = analysis.merge(
    spending,
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

analysis = analysis.rename(
    columns={
        "total_amount": "fundraising",
        "total_contribution_count": "contribution_count",
    }
)

analysis["viable_int"] = (
    analysis["is_viable"]
    .astype(bool)
    .astype(int)
)

analysis["log_fundraising"] = np.log1p(
    analysis["fundraising"]
)

analysis["log_spending"] = np.log1p(
    analysis["total_spending"]
)

print("Candidates on ballot:", len(analysis))
print("With fundraising:", analysis["fundraising"].notna().sum())
print("With spending:", analysis["total_spending"].notna().sum())


Candidates on ballot: 98
With fundraising: 65
With spending: 65


In [39]:
display(
    analysis[
        [
            "district",
            "canonical_candidate",
            "fundraising",
            "total_spending",
            "mentions",
            "first_place_votes",
            "is_viable",
        ]
    ]
    .sort_values(
        ["district", "mentions"],
        ascending=[True, False],
    )
)


,district,canonical_candidate,fundraising,total_spending,mentions,first_place_votes,is_viable
14,1,Candace Avalos,58893.80,177221.93,22267.0,8297.0,True
13,1,Steph Routh,76006.71,201809.37,20440.0,3894.0,True
6,1,Jamie Dunphy,35174.53,NaN,18470.0,5064.0,True
2,1,Loretta Smith,43744.96,1464141.21,17984.0,5586.0,True
0,1,Timur Ender,61809.84,157642.42,16858.0,3550.0,True
...,...,...,...,...,...,...,...
68,4,Brandon Farley,NaN,NaN,866.0,168.0,False
77,4,Tony Schwartz,NaN,NaN,856.0,100.0,False
85,4,Patrick Cashman,NaN,NaN,758.0,101.0,False
81,4,Lee Odell,NaN,NaN,639.0,100.0,False


## 3. Correlations: is there a relationship at all?

- **Pearson** asks how close the relationship is to a straight line.
- **Spearman** asks whether candidates with more money generally rank higher
  in support, even if the shape is curved.


In [40]:
money_measures = [
    "fundraising",
    "total_spending",
]

outcomes = [
    "mentions",
    "first_place_votes",
]

rows = []

for money in money_measures:
    for outcome in outcomes:
        pair = analysis[
            [money, outcome]
        ].dropna()

        rows.append(
            {
                "money": money,
                "outcome": outcome,
                "n": len(pair),
                "pearson": pair[money].corr(
                    pair[outcome]
                ),
                "spearman": pair[money].corr(
                    pair[outcome],
                    method="spearman",
                ),
            }
        )

correlations = pd.DataFrame(rows)

display(
    correlations.round(3)
)


,money,outcome,n,pearson,spearman
0,fundraising,mentions,65,0.779,0.800
1,fundraising,first_place_votes,65,0.752,0.764
2,total_spending,mentions,65,0.448,0.800
3,total_spending,first_place_votes,65,0.507,0.803


### Finding from the current run

All four relationships are positive. Fundraising has a particularly strong
relationship with ballot mentions (Pearson about **0.78** in the current run).

Spending has a much larger Spearman than Pearson correlation with mentions,
which is a clue that its relationship is not well represented by a straight
line in raw dollars. fileciteturn54file4


## 4. Raw dollars or log dollars?

We fit the same outcome two ways:

`support = a + b × dollars`

`support = a + b × log(1 + dollars)`

We compare R² **within the same finance measure and outcome**.


In [41]:
def fit_one_model(data, predictor, outcome, label):
    pair = data[
        [predictor, outcome]
    ].dropna()

    x = pair[predictor].to_numpy(dtype=float)
    y = pair[outcome].to_numpy(dtype=float)

    model = sm.OLS(
        y,
        sm.add_constant(x),
    ).fit()

    return {
        "outcome": outcome,
        "predictor": label,
        "n": int(model.nobs),
        "slope": model.params[1],
        "p_value": model.pvalues[1],
        "r_squared": model.rsquared,
    }


specifications = [
    ("fundraising", "fundraising — raw dollars"),
    ("log_fundraising", "fundraising — log dollars"),
    ("total_spending", "spending — raw dollars"),
    ("log_spending", "spending — log dollars"),
]

model_rows = []

for outcome in outcomes:
    for predictor, label in specifications:
        model_rows.append(
            fit_one_model(
                analysis,
                predictor,
                outcome,
                label,
            )
        )

models = pd.DataFrame(model_rows)

display(
    models
    .sort_values(
        ["outcome", "r_squared"],
        ascending=[True, False],
    )
    .round(
        {
            "slope": 3,
            "p_value": 6,
            "r_squared": 3,
        }
    )
)


,outcome,predictor,n,slope,p_value,r_squared
4,first_place_votes,fundraising — raw dollars,65,0.126,0.000000,0.565
7,first_place_votes,spending — log dollars,65,1740.274,0.000000,0.364
5,first_place_votes,fundraising — log dollars,65,1794.346,0.000000,0.336
6,first_place_votes,spending — raw dollars,65,0.011,0.000017,0.257
0,mentions,fundraising — raw dollars,65,0.354,0.000000,0.607
3,mentions,spending — log dollars,65,5477.457,0.000000,0.484
1,mentions,fundraising — log dollars,65,5853.901,0.000000,0.483
2,mentions,spending — raw dollars,65,0.026,0.000184,0.200


### Finding from the current run

The important correction is:

- **fundraising:** raw dollars fit better than log dollars;
- **spending:** log dollars fit better than raw dollars.

For mentions, the current R² values are about **0.607 vs 0.483** for
fundraising and **0.484 vs 0.200** for spending. fileciteturn54file5

That means later fundraising residuals should not say they use log money
because “Notebook 10 chose log.” It did not.


## 5. Interactive candidate explorer: fundraising and mentions

The regression result above uses **raw fundraising**. This figure therefore
keeps raw dollars on the x-axis and adds the same simple OLS trendline.

Hover over a point to identify the candidate.


In [42]:
plot_data = analysis.dropna(
    subset=["fundraising", "mentions"]
).copy()

plot_data["district"] = (
    plot_data["district"]
    .astype(str)
)

fig = px.scatter(
    plot_data,
    x="fundraising",
    y="mentions",
    color="district",
    symbol="is_viable",
    hover_name="canonical_candidate",
    hover_data={
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "contribution_count": ":,.0f",
        "is_viable": True,
    },
    trendline="ols",
    labels={
        "fundraising": "Total fundraising ($)",
        "mentions": "Ballot mentions",
        "district": "District",
        "is_viable": "Viable",
    },
    title="2024 City Council — fundraising and ballot mentions",
)

fig.show()


## 6. Interactive candidate explorer: spending and mentions

For spending, the log specification fits better. The x-axis is therefore shown
on a log scale for exploration.


In [43]:
plot_data = analysis.dropna(
    subset=["total_spending", "mentions"]
).copy()

plot_data["district"] = (
    plot_data["district"]
    .astype(str)
)

fig = px.scatter(
    plot_data,
    x="total_spending",
    y="mentions",
    color="district",
    symbol="is_viable",
    hover_name="canonical_candidate",
    hover_data={
        "total_spending": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "is_viable": True,
    },
    log_x=True,
    labels={
        "total_spending": "Reported spending ($, log scale)",
        "mentions": "Ballot mentions",
        "district": "District",
        "is_viable": "Viable",
    },
    title="2024 City Council — spending and ballot mentions",
)

fig.show()


## 7. Does the relationship differ by district?

A pooled citywide correlation can hide different district stories.


In [44]:
district_rows = []

for district in sorted(
    analysis["district"].dropna().unique()
):
    district_data = analysis[
        analysis["district"].eq(district)
    ]

    for money in money_measures:
        for outcome in outcomes:
            pair = district_data[
                [money, outcome]
            ].dropna()

            if len(pair) < 3:
                continue

            district_rows.append(
                {
                    "district": district,
                    "money": money,
                    "outcome": outcome,
                    "n": len(pair),
                    "pearson": pair[money].corr(
                        pair[outcome]
                    ),
                    "spearman": pair[money].corr(
                        pair[outcome],
                        method="spearman",
                    ),
                }
            )

district_correlations = pd.DataFrame(
    district_rows
)

display(
    district_correlations.round(3)
)


,district,money,outcome,n,pearson,spearman
0,1,fundraising,mentions,13,0.919,0.923
1,1,fundraising,first_place_votes,13,0.758,0.753
2,1,total_spending,mentions,11,0.436,0.927
3,1,total_spending,first_place_votes,11,0.471,0.836
4,2,fundraising,mentions,20,0.731,0.774
5,2,fundraising,first_place_votes,20,0.651,0.753
6,2,total_spending,mentions,19,0.750,0.744
7,2,total_spending,first_place_votes,19,0.501,0.737
8,3,fundraising,mentions,15,0.951,0.871
9,3,fundraising,first_place_votes,15,0.933,0.761


In [45]:
fig = px.bar(
    district_correlations,
    x="district",
    y="pearson",
    color="money",
    facet_col="outcome",
    barmode="group",
    hover_data=["n", "spearman"],
    labels={
        "district": "District",
        "pearson": "Pearson correlation",
        "money": "Finance measure",
    },
    title="Finance–support relationships differ across districts",
)

fig.show()


## 8. Conclusion

**What we learn from the current run:**

1. More campaign money is associated with more electoral support.
2. Fundraising relates somewhat more strongly to broad ballot mentions than
   to first-place votes.
3. Raw fundraising dollars fit the data better than log fundraising, while
   spending shows the opposite pattern.
4. The relationship differs by district; D4 is weaker for fundraising than
   D1 and D3 in the current data.
5. The regressions describe the **65 candidates with matched fundraising
   data**. They should not be described as estimates for all 98 candidates.

**Next question:** does the *structure* of finance add information once total
dollars are already in the model?


## 9. Export reusable tables

In [46]:
output_dir = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "topline"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

analysis.to_csv(
    output_dir / "candidate_topline_finance_support.csv",
    index=False,
)

correlations.to_csv(
    output_dir / "topline_correlations.csv",
    index=False,
)

models.to_csv(
    output_dir / "topline_raw_vs_log_ols.csv",
    index=False,
)

district_correlations.to_csv(
    output_dir / "district_finance_support_correlations.csv",
    index=False,
)

print("SAVED:", output_dir)


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/topline
